# Klasifikasi Kematangan Pisang Menggunakan Bayes Gaussian Classifier

Notebook ini dirancang untuk dijalankan di **Google Colab** menggunakan dataset yang **sudah difilter** (`BANANA-WHITE-BG`). Notebook ini fokus pada penggunaan **Bayes Gaussian Classifier** yang terbukti memberikan performa optimal pada dataset ini. Langkah-langkahnya meliputi:
1. **Kloning Repositori:** Mengkloning repositori GitHub untuk mengakses folder dataset terfilter (`BANANA-WHITE-BG`).
2. **Ekstraksi Fitur Optimal (8D):** Mengekstraksi statistik warna LAB dan HSV (kecerahan, kroma, saturasi).
3. **Pelatihan & Evaluasi Model:** Melatih dan menguji Bayes Gaussian Classifier.
4. **Visualisasi Hasil:** Menampilkan Confusion Matrix rata-rata dan batas keputusan (Decision Boundary) 2D hasil proyeksi PCA.

## Tahap 1: Setup Direktori & Clone Repositori Git

Jalankan cell di bawah ini untuk mengkloning repositori GitHub Anda ke Google Colab.

In [ ]:
# @title Kloning Repositori GitHub
# Kloning repositori proyek
!git clone https://github.com/Velazta/fruit-ripeness-detection.git

# Pindah ke direktori repositori
%cd fruit-ripeness-detection

## Tahap 2: Definisi Feature Extraction & Bayes Classifier

Pada tahap ini, kita mendefinisikan pembaca gambar, ekstraksi fitur 8D (LAB + HSV), serta implementasi Bayes Gaussian Classifier.

In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Path ke dataset terfilter di dalam repositori yang dikloning
DATASET_DIR = 'BANANA-WHITE-BG'

# Kategori pada dataset terfilter
CATEGORIES = {
    'rotten': 'rotten',
    'unripe': 'unripe',
    'ripe': 'ripe'
}

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff')

def extract_features(image_path, chroma_threshold=10):
    """
    Ekstraksi Fitur 8D Logis & Segmentasi Kroma LAB:
    - Segmentasi membuang background netral (hitam, putih, abu-abu).
    - Saluran LAB: L (kecerahan), A (hijau-merah), B (kuning-biru).
    - Saluran HSV: S (kejenuhan warna segar vs kusam).
    - Menghasilkan vektor 8-dimensi: [mean_L, mean_A, mean_B, mean_S, std_L, std_A, std_B, std_S]
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Jarak kroma dari netral (128, 128)
    a_channel = img_lab[:, :, 1].astype(float)
    b_channel = img_lab[:, :, 2].astype(float)
    chroma_dist = np.sqrt((a_channel - 128)**2 + (b_channel - 128)**2)
    
    # Buat mask segmentasi
    mask = chroma_dist > chroma_threshold
    if np.sum(mask) == 0:
        mask = np.ones_like(a_channel, dtype=bool)
        
    lab_pixels = img_lab[mask]
    hsv_pixels = img_hsv[mask]
    
    mean_lab = np.mean(lab_pixels, axis=0) # [mean_L, mean_A, mean_B]
    std_lab = np.std(lab_pixels, axis=0)   # [std_L, std_A, std_B]
    
    mean_s = np.mean(hsv_pixels[:, 1])
    std_s = np.std(hsv_pixels[:, 1])
    
    return np.array([
        mean_lab[0], mean_lab[1], mean_lab[2], mean_s,
        std_lab[0], std_lab[1], std_lab[2], std_s
    ])

def get_image_paths(category_name, split='train'):
    folder_path = os.path.join(DATASET_DIR, split, CATEGORIES[category_name])
    paths = []
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")
    for file in os.listdir(folder_path):
        if file.lower().endswith(IMAGE_EXTENSIONS):
            paths.append(os.path.join(folder_path, file))
    return paths

In [ ]:
class BayesClassifier:
    def __init__(self, reg=1e-5):
        self.means = {}
        self.covs = {}
        self.priors = {}
        self.reg = reg
    def fit(self, X_dict):
        total_samples = sum(len(X) for X in X_dict.values())
        for label, X in X_dict.items():
            self.means[label] = np.mean(X, axis=0)
            cov = np.cov(X, rowvar=False)
            self.covs[label] = cov + self.reg * np.eye(cov.shape[0])
            self.priors[label] = len(X) / total_samples
    def predict_single(self, x):
        log_posteriors = {}
        for label in self.means.keys():
            mean = self.means[label]
            cov = self.covs[label]
            prior = self.priors[label]
            diff = x - mean
            sign, logdet = np.linalg.slogdet(cov)
            if sign <= 0:
                logdet = np.sum(np.log(np.diagonal(cov)))
            inv_cov = np.linalg.inv(cov)
            mahalanobis_sq = diff.T @ inv_cov @ diff
            log_likelihood = -0.5 * logdet - 0.5 * mahalanobis_sq
            log_posteriors[label] = log_likelihood + np.log(prior)
        pred = max(log_posteriors, key=log_posteriors.get)
        return pred, log_posteriors
    def predict(self, X):
        return np.array([self.predict_single(x)[0] for x in X])

## Tahap 3: Pelatihan Model & Evaluasi

Menjalankan pipeline pelatihan sebanyak 3 kali run (menggunakan seed acak) untuk mendapatkan hasil agregat, serta menampilkan matriks kebingungan dan batas keputusan.

In [ ]:
def train_and_eval():
    all_train_paths = {}
    all_test_paths = {}
    for cat in CATEGORIES.keys():
        all_train_paths[cat] = get_image_paths(cat, 'train')
        all_test_paths[cat] = get_image_paths(cat, 'test')
        print(f"Kategori '{cat}': Train={len(all_train_paths[cat])} gambar, Test={len(all_test_paths[cat])} gambar")

    runs_results = []
    num_runs = 3
    seeds = [random.randint(1, 1000000) for _ in range(num_runs)]
    print(f"\nSeed acak yang dihasilkan untuk 3 kali run: {seeds}")

    cm_accumulator = np.zeros((3, 3))
    last_run_data = {}

    for run_idx, seed in enumerate(seeds):
        run_num = run_idx + 1
        print(f"\n=================== RUN KE-{run_num} (Seed: {seed}) ===================")
        random.seed(seed)
        np.random.seed(seed)

        train_samples = {cat: random.sample(all_train_paths[cat], 100) for cat in CATEGORIES.keys()}
        test_samples = {cat: random.sample(all_test_paths[cat], min(100, len(all_test_paths[cat]))) for cat in CATEGORIES.keys()}

        X_train_dict = {cat: [] for cat in CATEGORIES.keys()}
        X_test_dict = {cat: [] for cat in CATEGORIES.keys()}
        
        for cat in CATEGORIES.keys():
            for p in train_samples[cat]:
                X_train_dict[cat].append(extract_features(p, 10))
            X_train_dict[cat] = np.array(X_train_dict[cat])
            
            for p in test_samples[cat]:
                X_test_dict[cat].append(extract_features(p, 10))
            X_test_dict[cat] = np.array(X_test_dict[cat])

        X_train, y_train = [], []
        for cat in CATEGORIES.keys():
            X_train.extend(X_train_dict[cat])
            y_train.extend([cat] * len(X_train_dict[cat]))
        X_train = np.array(X_train)
        y_train = np.array(y_train)

        X_test, y_test = [], []
        for cat in CATEGORIES.keys():
            X_test.extend(X_test_dict[cat])
            y_test.extend([cat] * len(X_test_dict[cat]))
        X_test = np.array(X_test)
        y_test = np.array(y_test)

        if run_num == num_runs:
            last_run_data = {
                'X_train': X_train,
                'y_train': y_train,
                'X_test': X_test,
                'y_test': y_test,
                'X_train_dict': X_train_dict,
                'X_test_dict': X_test_dict
            }

        clf = BayesClassifier()
        clf.fit(X_train_dict)
        y_pred = clf.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, labels=list(CATEGORIES.keys()))
        macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', labels=list(CATEGORIES.keys()))
        cm = confusion_matrix(y_test, y_pred, labels=list(CATEGORIES.keys()))
        
        cm_accumulator += cm
        
        print(f"  Bayes Gaussian Classifier -> Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}")
        
        runs_results.append({
            'run': run_num,
            'seed': seed,
            'accuracy': acc,
            'macro_f1': macro_f1,
            'macro_precision': macro_prec,
            'macro_recall': macro_rec
        })

    print("\n=================== RINGKASAN AGREGAT (3 RUNS) ===================")
    accs = [r['accuracy'] for r in runs_results]
    f1s = [r['macro_f1'] for r in runs_results]
    precs = [r['macro_precision'] for r in runs_results]
    recs = [r['macro_recall'] for r in runs_results]
    
    mean_acc, std_acc = np.mean(accs), np.std(accs)
    mean_f1, std_f1 = np.mean(f1s), np.std(f1s)
    
    print(f"Bayes Gaussian Classifier Agregat:")
    print(f"  Akurasi      : {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  Macro F1-skor: {mean_f1:.4f} ± {std_f1:.4f}")

    # ---- PLOT CONFUSION MATRIX ----
    plt.figure(figsize=(6, 5))
    cm_avg = cm_accumulator / num_runs
    im = plt.imshow(cm_avg, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f"Bayes Gaussian Classifier\nRata-rata Akurasi: {mean_acc:.2%}")
    plt.colorbar(im)
    tick_marks = np.arange(len(CATEGORIES))
    plt.xticks(tick_marks, list(CATEGORIES.keys()))
    plt.yticks(tick_marks, list(CATEGORIES.keys()))
    thresh = cm_avg.max() / 2.
    for i in range(cm_avg.shape[0]):
        for j in range(cm_avg.shape[1]):
            plt.text(j, i, f"{cm_avg[i, j]:.1f}", ha="center", va="center", color="white" if cm_avg[i, j] > thresh else "black")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    # ---- PLOT DECISION BOUNDARY ----
    X_train_last = last_run_data['X_train']
    X_train_dict_last = last_run_data['X_train_dict']
    pca = PCA(n_components=2)
    X_train_pca = pca.fit_transform(X_train_last)
    X_train_pca_dict = {}
    idx = 0
    for cat in CATEGORIES.keys():
        n_samples = len(X_train_dict_last[cat])
        X_train_pca_dict[cat] = X_train_pca[idx : idx + n_samples]
        idx += n_samples
    
    clf_2d = BayesClassifier()
    clf_2d.fit(X_train_pca_dict)
    
    x_min, x_max = X_train_pca[:, 0].min() - 10, X_train_pca[:, 0].max() + 10
    y_min, y_max = X_train_pca[:, 1].min() - 10, X_train_pca[:, 1].max() + 10
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 1.0), np.arange(y_min, y_max, 1.0))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    pred_grid = clf_2d.predict(grid_points)
    
    label_to_int = {'rotten': 0, 'unripe': 1, 'ripe': 2}
    Z = np.array([label_to_int[p] for p in pred_grid]).reshape(xx.shape)
    
    plt.figure(figsize=(7, 6))
    from matplotlib.colors import ListedColormap
    bg_cmap = ListedColormap(['#E5D4C0', '#D1E7DD', '#FCE8E6'])
    point_colors = {'rotten': '#8B5A2B', 'unripe': '#1E4620', 'ripe': '#B03A2E'}
    
    plt.contourf(xx, yy, Z, alpha=0.6, cmap=bg_cmap)
    for cat in CATEGORIES.keys():
        pts = X_train_pca_dict[cat]
        plt.scatter(pts[:, 0], pts[:, 1], c=point_colors[cat], label=cat, edgecolors='black', s=40)
    plt.title('Bayes Gaussian Classifier Decision Boundary (PCA)')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend(loc='lower left')
    plt.tight_layout()
    plt.show()

if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
    train_and_eval()
else:
    print("Dataset terfilter BANANA-WHITE-BG tidak ditemukan. Pastikan folder/repositori sudah di-clone.")